# Setup

## Install necessary libraries

In [ ]:
!pip install google-adk -q
!pip install litellm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.3/239.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.7/335.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.5/158.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## Configuration - API Keys

In [ ]:
import os

from google.colab import userdata

# Get the OpenAI API key from environment variables
openai_api_key = os.environ.get("OPENAI_API_KEY")

# Gemini API Key (Get from Google AI Studio: https://aistudio.google.com/app/apikey)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# OpenAI API Key (Get from OpenAI Platform: https://platform.openai.com/api-keys)
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Configure ADK to use API keys directly (not Vertex AI for this setup)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


In [ ]:
# @title Import necessary libraries
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.tools import FunctionTool


import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries imported.")

Libraries imported.


In [ ]:
# @title Define Model Constants for easier use

# More supported models can be referenced here: https://ai.google.dev/gemini-api/docs/models#model-variations
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

# More supported models can be referenced here: https://docs.litellm.ai/docs/providers/openai#openai-chat-completion-models
MODEL_GPT_4O = "openai/gpt-4.1" # You can also try: gpt-4.1-mini, gpt-4o etc.

# More supported models can be referenced here: https://docs.litellm.ai/docs/providers/anthropic
MODEL_CLAUDE_SONNET = "anthropic/claude-sonnet-4-20250514" # You can also try: claude-opus-4-20250514 , claude-3-7-sonnet-20250219 etc

print("\nEnvironment configured.")


Environment configured.


# Basic Agent

## Define the agent

In [ ]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

MODEL = MODEL_GEMINI_2_0_FLASH

# Define a basic agent with a simple instruction
basic_agent = Agent(
    name="logging_agent_v1",
    model=MODEL,
    instruction="You are a helpful assistant that responds to user queries."
)

print(f"Agent '{basic_agent.name}' created using model '{MODEL}'.")

Agent 'logging_agent_v1' created using model 'gemini-2.0-flash'.


## Setup Session Service and APP for the basic agent

In [ ]:
# Setup Session Service and Runner for the basic agent
session_service = InMemorySessionService()
APP_NAME = "basic_agent_app"

## For user 1, setup runner and session 1

In [ ]:
# --- Session 1 ---
USER_ID_1 = "user_1"
SESSION_ID_1 = "session_001"

session_1 = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID_1,
    session_id=SESSION_ID_1
)
print(f"\nSession created for logging agent (Session 1): App='{APP_NAME}', User='{USER_ID_1}', Session='{SESSION_ID_1}'")

runner_1 = Runner(
    agent=basic_agent,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner created for logging agent '{runner_1.agent.name}' (Session 1).")


Session created for logging agent (Session 1): App='basic_agent_app', User='user_1', Session='session_001'
Runner created for logging agent 'logging_agent_v1' (Session 1).


## For user 2, setup runner and session 1

In [ ]:

# --- Session 2 ---
USER_ID_2 = "user_2"
SESSION_ID_2 = "session_002"

session_2 = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID_2,
    session_id=SESSION_ID_2
)
print(f"\nSession created for logging agent (Session 2): App='{APP_NAME}', User='{USER_ID_2}', Session='{SESSION_ID_2}'")

runner_2 = Runner(
    agent=basic_agent, # Using the same agent instance
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner created for logging agent '{runner_2.agent.name}' (Session 2).")


Session created for logging agent (Session 2): App='basic_agent_app', User='user_2', Session='session_002'
Runner created for logging agent 'logging_agent_v1' (Session 2).


## Define function to call the agent

In [ ]:
async def call_agent(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response."

  # Run the agent and get the final response
  # This iterates through events and waits for the one marked as final
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      if event.is_final_response():
          if event.content and event.content.parts:
             final_response_text = event.content.parts[0].text
          elif event.error_message:
             final_response_text = f"Agent error: {event.error_message}"
          break # Stop after the final response

  print(f"<<< Agent Response: {final_response_text}")


## Define function to call the agent and print all events

In [ ]:

async def call_agent_with_logging(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints all generated events."""

  print(f"\n>>> User Query ({user_id}/{session_id}): {query}")

  content = types.Content(role='user', parts=[types.Part(text=query)])

  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}")
      if event.content and event.content.parts:
          print(f"    Content Text: {event.content.parts[0].text[:100]}...")
      if event.actions:
          print(f"    Actions: {event.actions}")
      if event.error_message:
          print(f"    Error: {event.error_message}")

## Inference

In [ ]:
# Run sample conversations without logging for both sessions
async def run_conversations():
    print("\n--- Starting Agent Conversation (User 1, Session 1) ---")
    await call_agent("Hello, how are you doing today?",
                                       runner=runner_1,
                                       user_id=USER_ID_1,
                                       session_id=SESSION_ID_1)

    await call_agent("What is the capital of France?",
                                       runner=runner_1,
                                       user_id=USER_ID_1,
                                       session_id=SESSION_ID_1)

    print("\n--- Starting Agent Conversation (User 2, Session 2) ---")
    await call_agent("Hi there, what's the weather like?",
                                       runner=runner_2,
                                       user_id=USER_ID_2,
                                       session_id=SESSION_ID_2)

    await call_agent("Tell me a fun fact.",
                                       runner=runner_2,
                                       user_id=USER_ID_2,
                                       session_id=SESSION_ID_2)


# Execute the conversation with logging
await run_conversations()


--- Starting Agent Conversation (User 1, Session 1) ---

>>> User Query: Hello, how are you doing today?
<<< Agent Response: I'm doing well, thank you for asking! How can I help you today?


>>> User Query: What is the capital of France?
<<< Agent Response: The capital of France is Paris.


--- Starting Agent Conversation (User 2, Session 2) ---

>>> User Query: Hi there, what's the weather like?
<<< Agent Response: I am sorry, I do not have the current weather information. To provide you with accurate weather details, I need to know your location. Could you please tell me where you are or where you would like to know the weather for?


>>> User Query: Tell me a fun fact.
<<< Agent Response: Okay, here's a fun fact for you:

Did you know that a group of owls is called a parliament? 



## Inference (with event logs)

In [ ]:
# Run sample conversations with logging for both sessions
async def run_logging_conversations():
    print("\n--- Starting Agent Conversation (User 1, Session 1) ---")
    await call_agent_with_logging("Hello, how are you doing today?",
                                       runner=runner_1,
                                       user_id=USER_ID_1,
                                       session_id=SESSION_ID_1)

    await call_agent_with_logging("What is the capital of France?",
                                       runner=runner_1,
                                       user_id=USER_ID_1,
                                       session_id=SESSION_ID_1)

    print("\n--- Starting Agent Conversation (User 2, Session 2) ---")
    await call_agent_with_logging("Hi there, what's the weather like?",
                                       runner=runner_2,
                                       user_id=USER_ID_2,
                                       session_id=SESSION_ID_2)

    await call_agent_with_logging("Tell me a fun fact.",
                                       runner=runner_2,
                                       user_id=USER_ID_2,
                                       session_id=SESSION_ID_2)


# Execute the conversation with logging
await run_logging_conversations()


--- Starting Agent Conversation (User 1, Session 1) ---

>>> User Query (user_1/session_001): Hello, how are you doing today?
  [Event] Author: logging_agent_v1, Type: Event, Final: True
    Content Text: I'm doing well, thank you! How can I help you?
...
    Actions: skip_summarization=None state_delta={} artifact_delta={} transfer_to_agent=None escalate=None requested_auth_configs={}

>>> User Query (user_1/session_001): What is the capital of France?
  [Event] Author: logging_agent_v1, Type: Event, Final: True
    Content Text: The capital of France is Paris.
...
    Actions: skip_summarization=None state_delta={} artifact_delta={} transfer_to_agent=None escalate=None requested_auth_configs={}

--- Starting Agent Conversation (User 2, Session 2) ---

>>> User Query (user_2/session_002): Hi there, what's the weather like?
  [Event] Author: logging_agent_v1, Type: Event, Final: True
    Content Text: I need to know your location to provide you with weather information. Could you plea

## Inspect Sessions

In [ ]:
# Fetch a specific session using its identifiers
fetched_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID_1,
    session_id=SESSION_ID_1
)

print("\n--- Fetched Session Details ---")
print(f"Session ID: {fetched_session.id}")
print(f"App Name: {fetched_session.app_name}")
print(f"User ID: {fetched_session.user_id}")
print(f"Number of events: {len(fetched_session.events)}")
# You can access the conversation history (events) like this:
# for event in fetched_session.events:
#     print(event)

# Fetch the second session as well
fetched_session_2 = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID_2,
    session_id=SESSION_ID_2
)

print("\n--- Fetched Second Session Details ---")
print(f"Session ID: {fetched_session_2.id}")
print(f"User ID: {fetched_session_2.user_id}")
print(f"Number of events: {len(fetched_session_2.events)}")


--- Fetched Session Details ---
Session ID: session_001
App Name: basic_agent_app
User ID: user_1
Number of events: 8

--- Fetched Second Session Details ---
Session ID: session_002
User ID: user_2
Number of events: 8


# Using LiteLLM in ADK

In [ ]:
# @title Define the Joke Agent and Tool
openai_api_key = os.environ.get("OPENAI_API_KEY")

# Set up the model using LiteLLM
# Make sure the model name is supported by your LiteLLM setup
# Using "gpt-3.5-turbo" from LiteLLM here
model = LiteLlm(
    model="gpt-3.5-turbo",
    api_key=openai_api_key
)

def get_ai_engineer_joke() -> dict:
    """
    Returns a random AI engineer joke from a curated list.
    Use this when the user asks for an AI-related joke.
    """
    jokes = [
        "Why did the neural network break up with the decision tree? It couldn’t handle its lack of depth.",
        "Why do AI engineers make terrible secret agents? Because they always leave a trace!",
        "How does an AI tell someone they’re wrong? With a high confidence score and low accuracy.",
        "Why was the LLM always tired? It had too many parameters to sleep on.",
        "How do you comfort an overfitting model? You tell it, 'It's not your data, it's your complexity.'",
        "Why did the AI fail its driving test? It couldn’t handle unstructured environments.",
        "Why do transformers love attention? Because it’s all they need!",
        "Why don’t AI models gossip? Because their training data is confidential.",
        "Why did the engineer talk to the model during inference? Because they misunderstood 'zero-shot'.",
        "Why was the dataset feeling down? It needed some normalization."
    ]
    selected_joke = random.choice(jokes)
    return {
        "joke": selected_joke
    }

ai_engineer_joke_agent = Agent(
    name="ai_engineer_joke_agent",
    model=model, # Use the LiteLlm model
    description="An agent that tells AI engineer jokes",
    instruction="You are a witty assistant that tells jokes only about AI engineers, models, and data. Use only the tool get_ai_engineer_joke to tell a joke.",
    tools=[FunctionTool(get_ai_engineer_joke)]
)

print(f"Joke Agent '{ai_engineer_joke_agent.name}' created using LiteLLM model.")

Joke Agent 'ai_engineer_joke_agent' created using LiteLLM model.


In [ ]:
# @title Setup Session Service and Runner for Joke Agent

# Reuse the InMemorySessionService instance or create a new one if needed
# session_service = InMemorySessionService() # Uncomment if starting fresh
# Assuming session_service is already initialized from previous cells

# Define constants for identifying the interaction context for the joke agent
JOKE_APP_NAME = "joke_agent_app"
JOKE_USER_ID = "joke_user_1"
JOKE_SESSION_ID = "joke_session_001" # Using a fixed ID for simplicity

# Create a session for the joke agent
# Assuming session_service is accessible
try:
    joke_session = await session_service.create_session(
        app_name=JOKE_APP_NAME,
        user_id=JOKE_USER_ID,
        session_id=JOKE_SESSION_ID
    )
    print(f"Session created for joke agent: App='{JOKE_APP_NAME}', User='{JOKE_USER_ID}', Session='{JOKE_SESSION_ID}'")
except Exception as e:
    print(f"Error creating session: {e}")
    # If session_service was not initialized, create a new one
    print("Initializing a new InMemorySessionService...")
    session_service = InMemorySessionService()
    joke_session = await session_service.create_session(
        app_name=JOKE_APP_NAME,
        user_id=JOKE_USER_ID,
        session_id=JOKE_SESSION_ID
    )
    print(f"New Session created for joke agent: App='{JOKE_APP_NAME}', User='{JOKE_USER_ID}', Session='{JOKE_SESSION_ID}'")


# Create a runner for the joke agent
joke_runner = Runner(
    agent=ai_engineer_joke_agent, # The joke agent we want to run
    app_name=JOKE_APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for joke agent '{joke_runner.agent.name}'.")

Session created for joke agent: App='joke_agent_app', User='joke_user_1', Session='joke_session_001'
Runner created for joke agent 'ai_engineer_joke_agent'.


In [ ]:
# @title Call the Joke Agent

# Assuming joke_runner, JOKE_USER_ID, and JOKE_SESSION_ID are defined from the previous cell
# Assuming call_agent_async is defined from a previous cell, or define it here

# Define Agent Interaction Function if not already defined
# This is a simplified version that just gets the final response text
# For detailed logging, use the call_agent_with_logging function defined earlier
if 'call_agent_async' not in globals():
    from google.genai import types # For creating message Content/Parts
    async def call_agent_async(query: str, runner, user_id, session_id):
      """Sends a query to the agent and prints the final response."""
      print(f"\n>>> User Query: {query}")
      content = types.Content(role='user', parts=[types.Part(text=query)])
      final_response_text = "Agent did not produce a final response."
      async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
          if event.is_final_response():
              if event.content and event.content.parts:
                 final_response_text = event.content.parts[0].text
              elif event.error_message:
                 final_response_text = f"Agent error: {event.error_message}"
              break
      print(f"<<< Agent Response: {final_response_text}")


async def run_joke_conversation():
    print("\n--- Starting Joke Agent Conversation ---")

    # Query that should trigger the get_ai_engineer_joke tool
    await call_agent_async("Tell me an AI engineer joke.",
                                       runner=joke_runner,
                                       user_id=JOKE_USER_ID,
                                       session_id=JOKE_SESSION_ID)

    # Another query to see if it tells a different joke
    await call_agent_async("Got another AI joke?",
                                       runner=joke_runner,
                                       user_id=JOKE_USER_ID,
                                       session_id=JOKE_SESSION_ID)

    # Query that should NOT trigger the tool (not about AI jokes)
    await call_agent_async("Tell me a knock-knock joke.",
                                        runner=joke_runner,
                                        user_id=JOKE_USER_ID,
                                        session_id=JOKE_SESSION_ID)


    print("\n--- Joke Agent Conversation Finished ---")

# Execute the conversation
await run_joke_conversation()


--- Starting Joke Agent Conversation ---

>>> User Query: Tell me an AI engineer joke.
<<< Agent Response: How do you comfort an overfitting model? You tell it, "It's not your data, it's your complexity."

>>> User Query: Got another AI joke?
<<< Agent Response: How does an AI tell someone they’re wrong? With a high confidence score and low accuracy.

>>> User Query: Tell me a knock-knock joke.
<<< Agent Response: I specialize in AI engineer jokes. Would you like to hear another one?

--- Joke Agent Conversation Finished ---


# Tool-Calling Agent

## Using built-in tools

In [ ]:
# @title Define the Web Search Agent

from google.adk.tools import google_search

WEB_SEARCH_APP_NAME = "web_search_agent_app"
WEB_SEARCH_USER_ID = "web_user_1"
WEB_SEARCH_SESSION_ID = "web_search_session_001"

# Create the web search agent with the most explicit instructions
web_search_agent = Agent(
    name="web_search_agent",
    model="gemini-1.5-flash-latest",
    instruction=(
        "You are an expert assistant whose primary function is to use the 'web_search' tool to answer user questions that require current or external information. "
        "Your goal is to find and summarize relevant information from the web search results."
        "\n\n"
        "**CRITICAL INSTRUCTION: YOU MUST USE THE 'web_search' tool for ANY query asking about:**\n"
        "- **Current values or prices** (e.g., 'price of Bitcoin', 'stock quotes')\n"
        "- **Real-time conditions** (e.g., 'weather', 'traffic')\n"
        "- **Upcoming events, dates, or schedules**\n"
        "- **The absolute latest news or research findings**\n"
        "- **Any specific fact that changes frequently or is outside your training data cutoff.**\n"
        "\n"
        "**Strict Procedure for Relevant Queries:**\n"
        "1. **ANALYZE:** Carefully read the user's query and determine if it falls into any of the categories listed above that require web search.\n"
        "2. **FORMULATE:** Construct a precise and effective search query that will likely yield the information needed.\n"
        "3. **CALL TOOL:** Immediately call the 'web_search' tool using the formulated query. **UNDER NO CIRCUMSTANCES should you answer these types of questions using your internal knowledge or provide outdated information.**\n"
        "4. **PROCESS AND ANSWER:** Once the 'web_search' tool provides its output, you **MUST** process the results to formulate your final answer.\n"
        "   - If the tool reports 'status: success' and the 'results' list is NOT empty, read the 'title', 'link', and 'snippet' for each result.\n"
        "   - **SYNTHESIZE** the key information from the snippets to directly and accurately answer the user's original question.\n"
        "   - Your final response should be a concise summary based *only* on the information found in the tool's output.\n"
        "   - If the tool reports 'status: success' but the 'results' list IS empty or indicates no relevant results were found (e.g., 'No relevant results found.'), clearly state to the user that your search did not find information for their query.\n"
        "   - If the tool reports 'status: error', inform the user that an error occurred during the search attempt.\n"
        "\n"
        "For queries that are general knowledge and DO NOT require current/external information, you may answer without using the tool."
        "5. PROVIDE SOURCES: Once you fetch and format the responses, ensure you are also appending the sources to the response."
    ),
    tools=[google_search]
)

print(f"Web Search Agent '{web_search_agent.name}' created using model '{web_search_agent.model}' with final instructions.")



Web Search Agent 'web_search_agent' created using model 'gemini-1.5-flash-latest' with final instructions.


In [ ]:
# @title Define Session Service and Runner
session_service_new = InMemorySessionService()


NEW_SESSION_ID = "web_search_session_new_001"
web_search_session_new = await session_service_new.create_session(
    app_name=WEB_SEARCH_APP_NAME,
    user_id=WEB_SEARCH_USER_ID,
    session_id=WEB_SEARCH_SESSION_ID
)

print(f"NEW Session created for web search agent: App='{WEB_SEARCH_APP_NAME}', User='{WEB_SEARCH_USER_ID}', Session='{NEW_SESSION_ID}'")


web_search_runner_new = Runner(
    agent=web_search_agent,
    app_name=WEB_SEARCH_APP_NAME,
    session_service=session_service_new
)
print(f"NEW Runner created for agent '{web_search_runner_new.agent.name}'.")

NEW Session created for web search agent: App='web_search_agent_app', User='web_user_1', Session='web_search_session_new_001'
NEW Runner created for agent 'web_search_agent'.


In [ ]:
# @title Inference
async def run_web_search_conversation_retest_v3():

    print("\n--- Starting Web Search Conversation (New Session/Runner) ---")

    # Query that requires web search for current information (price)
    await call_agent_async("What is the current price of Ethereum?",
                                       runner=web_search_runner_new,
                                       user_id=WEB_SEARCH_USER_ID,
                                       session_id=WEB_SEARCH_SESSION_ID)

    # Query that requires web search for current information (another price)
    await call_agent_async("What is the current price of Bitcoin?",
                                       runner=web_search_runner_new,
                                       user_id=WEB_SEARCH_USER_ID,
                                       session_id=WEB_SEARCH_SESSION_ID)


    # Query for specific information likely not in training data (research)
    await call_agent_async("Tell me about the latest breakthroughs in fusion energy research.",
                                       runner=web_search_runner_new,
                                       user_id=WEB_SEARCH_USER_ID,
                                       session_id=WEB_SEARCH_SESSION_ID)

    # Query that might require confirming information (conference)
    await call_agent_async("When is the next major international climate conference scheduled?",
                                       runner=web_search_runner_new,
                                       user_id=WEB_SEARCH_USER_ID,
                                       session_id=WEB_SEARCH_SESSION_ID)


    print("\n--- Retest v3 Web Search Conversation Finished ---")

# Execute the conversation
await run_web_search_conversation_retest_v3()


--- Starting Web Search Conversation (New Session/Runner) ---

>>> User Query: What is the current price of Ethereum?


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

## Using custom tools

In [ ]:
# @title Define the get_weather Tool
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---")
    city_normalized = city.lower().replace(" ", "") # Basic standardization

    # Mock weather data
    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

# Example tool usage (optional test)
print(get_weather("New York"))
print(get_weather("Paris"))

--- Tool: get_weather called for city: New York ---
{'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25°C.'}
--- Tool: get_weather called for city: Paris ---
{'status': 'error', 'error_message': "Sorry, I don't have weather information for 'Paris'."}


In [ ]:
# @title Define the Weather Agent

AGENT_MODEL = MODEL_GEMINI_2_0_FLASH

weather_agent = Agent(
    name="weather_agent_v1",
    model=AGENT_MODEL,
    description="Provides weather information for specific cities.",
    instruction="You are a helpful weather assistant. "
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools=[get_weather], # Pass the custom tool function
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_agent_v1' created using model 'gemini-2.0-flash'.


In [ ]:
# @title Setup Session Service and Runner

session_service = InMemorySessionService()

APP_NAME = "weather_tutorial_app"
USER_ID = "user_1"
SESSION_ID = "session_001"

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='weather_tutorial_app', User='user_1', Session='session_001'
Runner created for agent 'weather_agent_v1'.


In [ ]:
# @title Define Agent Interaction Function

from google.genai import types

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default


  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # Uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate:
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break

  print(f"<<< Agent Response: {final_response_text}")

In [ ]:
# @title Inference


async def run_conversation():
    await call_agent_async("What is the weather like in London?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

    await call_agent_async("How about Bengaluru, India?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

    await call_agent_async("Tell me the weather in New York",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)


await run_conversation()




>>> User Query: What is the weather like in London?


--- Tool: get_weather called for city: London ---
<<< Agent Response: The weather in London is cloudy with a temperature of 15°C.


>>> User Query: How about Bengaluru, India?


--- Tool: get_weather called for city: Bengaluru ---
<<< Agent Response: I am sorry, I don't have weather information for Bengaluru.


>>> User Query: Tell me the weather in New York


--- Tool: get_weather called for city: New York ---
<<< Agent Response: The weather in New York is sunny with a temperature of 25°C.



## Limitations of a multi-tool agent


In [ ]:
# @title Define the multi-tool agent
multi_tool_agent = Agent(
    name="multi_tool_agent",
    model="gemini-1.5-flash-latest",
    instruction=(
        "You are a versatile assistant that can provide weather information and search the web. "
        "When the user asks for the weather in a specific city, use the 'get_weather' tool. "
        "For any other queries requiring current or external information (like prices, news, events), "
        "use the 'google_search' tool to find the answer. "
        "If a tool returns an error or finds no relevant information, inform the user politely."
    ),
    tools=[get_weather, google_search], # both tools
)

print(f"Agent '{multi_tool_agent.name}' created using model '{multi_tool_agent.model}' with multiple tools.")

Agent 'multi_tool_agent' created using model 'gemini-1.5-flash-latest' with multiple tools.


### Set up session service and runner



In [ ]:
session_service = InMemorySessionService()

MULTI_TOOL_APP_NAME = "multi_tool_agent_app"
MULTI_TOOL_USER_ID = "multi_tool_user_1"
MULTI_TOOL_SESSION_ID = "multi_tool_session_001"

multi_tool_session = await session_service.create_session(
    app_name=MULTI_TOOL_APP_NAME,
    user_id=MULTI_TOOL_USER_ID,
    session_id=MULTI_TOOL_SESSION_ID
)
print(f"Session created for multi-tool agent: App='{MULTI_TOOL_APP_NAME}', User='{MULTI_TOOL_USER_ID}', Session='{MULTI_TOOL_SESSION_ID}'")

multi_tool_runner = Runner(
    agent=multi_tool_agent,
    app_name=MULTI_TOOL_APP_NAME,
    session_service=session_service
)
print(f"Runner created for multi-tool agent '{multi_tool_runner.agent.name}'.")

Session created for multi-tool agent: App='multi_tool_agent_app', User='multi_tool_user_1', Session='multi_tool_session_001'
Runner created for multi-tool agent 'multi_tool_agent'.


### Inference



In [ ]:
async def run_multi_tool_conversation():
    print("\n--- Starting Multi-Tool Agent Conversation ---")

    # Query that should trigger the get_weather tool
    await call_agent_async("What is the weather like in Tokyo?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the google_search tool (latest news)
    await call_agent_async("What is the latest news on AI?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the google_search tool (upcoming event)
    await call_agent_async("When is the next major Formula 1 race?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the get_weather tool (another city)
    await call_agent_async("How about the weather in London?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that might be ambiguous (e.g., "Tell me about the climate in Paris") -
    # This might use google_search for general climate info or get_weather if interpreted as current.
    # Based on the prompt, it's more likely to use get_weather if the city is supported,
    # otherwise it might fall back to google_search for general climate info.
    # Let's test with Paris which is not in the mock weather data to see fallback.
    await call_agent_async("Tell me about the climate in Paris",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)


    print("\n--- Multi-Tool Agent Conversation Finished ---")

# Await the execution of the conversation function
await run_multi_tool_conversation()


--- Starting Multi-Tool Agent Conversation ---

>>> User Query: What is the weather like in Tokyo?


ValueError: Google search tool can not be used with other tools in Gemini 1.x.

In [ ]:
# Update the multi-tool agent to use a Gemini 2.0 model
multi_tool_agent = Agent(
    name="multi_tool_agent_v2",
    model=MODEL_GEMINI_2_0_FLASH,
    instruction=(
        "You are a versatile assistant that can provide weather information and search the web. "
        "When the user asks for the weather in a specific city, use the 'get_weather' tool. "
        "For any other queries requiring current or external information (like prices, news, events), "
        "use the 'google_search' tool to find the answer. "
        "If a tool returns an error or finds no relevant information, inform the user politely."
    ),
    tools=[get_weather, google_search], # Include both tools
)

print(f"Agent '{multi_tool_agent.name}' updated to use model '{multi_tool_agent.model}' with multiple tools.")

# Re-create the runner with the updated agent
multi_tool_runner = Runner(
    agent=multi_tool_agent,
    app_name=MULTI_TOOL_APP_NAME,
    session_service=session_service
)
print(f"Runner updated for agent '{multi_tool_runner.agent.name}'.")

# Redefine and re-run the asynchronous function for the multi-tool conversation
async def run_multi_tool_conversation():
    print("\n--- Starting Multi-Tool Agent Conversation ---")

    # Query that should trigger the get_weather tool
    await call_agent_async("What is the weather like in Tokyo?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the google_search tool (latest news)
    await call_agent_async("What is the latest news on AI?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the google_search tool (upcoming event)
    await call_agent_async("When is the next major Formula 1 race?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that should trigger the get_weather tool (another city)
    await call_agent_async("How about the weather in London?",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)

    # Query that might be ambiguous (e.g., "Tell me about the climate in Paris") -
    # This might use google_search for general climate info or get_weather if interpreted as current.
    # Based on the prompt, it's more likely to use get_weather if the city is supported,
    # otherwise it might fall back to google_search for general climate info.
    # Let's test with Paris which is not in the mock weather data to see fallback.
    await call_agent_async("Tell me about the climate in Paris",
                                       runner=multi_tool_runner,
                                       user_id=MULTI_TOOL_USER_ID,
                                       session_id=MULTI_TOOL_SESSION_ID)


    print("\n--- Multi-Tool Agent Conversation Finished ---")

# Await the execution of the conversation function
await run_multi_tool_conversation()

Agent 'multi_tool_agent_v2' updated to use model 'gemini-2.0-flash' with multiple tools.
Runner updated for agent 'multi_tool_agent_v2'.

--- Starting Multi-Tool Agent Conversation ---

>>> User Query: What is the weather like in Tokyo?


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Tool use with function calling is unsupported', 'status': 'INVALID_ARGUMENT'}}

## Multi-tool Agent


In [ ]:
# @title Define the Dictionary Lookup Tool
def dictionary_lookup(word: str) -> dict:
    """Looks up the definition of a word from a mock dictionary.

    Args:
        word (str): The word to look up.

    Returns:
        dict: A dictionary containing the lookup result.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'definition' key.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: dictionary_lookup called for word: {word} ---") # Log tool execution
    word_normalized = word.lower().strip()

    # Mock dictionary data
    mock_dict = {
        "apple": {"status": "success", "definition": "A round fruit with firm, white flesh and a green, red, or yellow skin."},
        "banana": {"status": "success", "definition": "A long, curved fruit which grows in clusters and has soft pulpy flesh and yellow skin when ripe."},
        "cherry": {"status": "success", "definition": "A small, soft round red or black fruit with a single hard stone in the middle."},
    }

    if word_normalized in mock_dict:
        return mock_dict[word_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I could not find the definition for '{word}'."}

print(dictionary_lookup("apple"))
print(dictionary_lookup("grape"))

--- Tool: dictionary_lookup called for word: apple ---
{'status': 'success', 'definition': 'A round fruit with firm, white flesh and a green, red, or yellow skin.'}
--- Tool: dictionary_lookup called for word: grape ---
{'status': 'error', 'error_message': "Sorry, I could not find the definition for 'grape'."}


In [ ]:
# @title Define the Text Manipulation Tool
def text_manipulation(text: str, operation: str) -> dict:
    """Performs simple text manipulation operations.

    Args:
        text (str): The input text string.
        operation (str): The operation to perform ('reverse', 'uppercase', 'lowercase').

    Returns:
        dict: A dictionary containing the result of the operation.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'result' key.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: text_manipulation called for text: '{text}', operation: '{operation}' ---") # Log tool execution
    operation_normalized = operation.lower().strip()

    try:
        if operation_normalized == "reverse":
            return {"status": "success", "result": text[::-1]}
        elif operation_normalized == "uppercase":
            return {"status": "success", "result": text.upper()}
        elif operation_normalized == "lowercase":
            return {"status": "success", "result": text.lower()}
        else:
            return {"status": "error", "error_message": f"Unsupported operation: '{operation}'. Supported operations are 'reverse', 'uppercase', 'lowercase'."}
    except Exception as e:
        return {"status": "error", "error_message": f"An error occurred during text manipulation: {e}"}

# Example tool usage (optional test)
print(text_manipulation("Hello World", "reverse"))
print(text_manipulation("Hello World", "UPPERCASE"))
print(text_manipulation("Hello World", "capitalize"))

--- Tool: text_manipulation called for text: 'Hello World', operation: 'reverse' ---
{'status': 'success', 'result': 'dlroW olleH'}
--- Tool: text_manipulation called for text: 'Hello World', operation: 'UPPERCASE' ---
{'status': 'success', 'result': 'HELLO WORLD'}
--- Tool: text_manipulation called for text: 'Hello World', operation: 'capitalize' ---
{'status': 'error', 'error_message': "Unsupported operation: 'capitalize'. Supported operations are 'reverse', 'uppercase', 'lowercase'."}


In [ ]:
# @title Define the multi-tool agent with custom tools
custom_multi_tool_agent = Agent(
    name="custom_multi_tool_agent",
    model="gemini-1.5-flash-latest",
    instruction=(
        "You are a helpful assistant equipped with custom tools. "
        "When the user asks for the definition of a word, use the 'dictionary_lookup' tool. "
        "When the user asks to perform an operation on text (like reverse, uppercase, or lowercase), use the 'text_manipulation' tool. "
        "If a tool returns an error or cannot fulfill the request, inform the user politely."
    ),
    tools=[dictionary_lookup, text_manipulation],
)

print(f"Agent '{custom_multi_tool_agent.name}' created using model '{custom_multi_tool_agent.model}' with custom tools.")

Agent 'custom_multi_tool_agent' created using model 'gemini-1.5-flash-latest' with custom tools.


In [ ]:
# @title Set up session service and runner

session_service = InMemorySessionService()

CUSTOM_MULTI_TOOL_APP_NAME = "custom_multi_tool_agent_app"
CUSTOM_MULTI_TOOL_USER_ID = "custom_multi_tool_user_1"
CUSTOM_MULTI_TOOL_SESSION_ID = "custom_multi_tool_session_001"

custom_multi_tool_session = await session_service.create_session(
    app_name=CUSTOM_MULTI_TOOL_APP_NAME,
    user_id=CUSTOM_MULTI_TOOL_USER_ID,
    session_id=CUSTOM_MULTI_TOOL_SESSION_ID
)
print(f"Session created for custom multi-tool agent: App='{CUSTOM_MULTI_TOOL_APP_NAME}', User='{CUSTOM_MULTI_TOOL_USER_ID}', Session='{CUSTOM_MULTI_TOOL_SESSION_ID}'")

custom_multi_tool_runner = Runner(
    agent=custom_multi_tool_agent,
    app_name=CUSTOM_MULTI_TOOL_APP_NAME,
    session_service=session_service
)
print(f"Runner created for custom multi-tool agent '{custom_multi_tool_runner.agent.name}'.")

Session created for custom multi-tool agent: App='custom_multi_tool_agent_app', User='custom_multi_tool_user_1', Session='custom_multi_tool_session_001'
Runner created for custom multi-tool agent 'custom_multi_tool_agent'.


In [ ]:
# @title Inference
async def run_custom_multi_tool_conversation():
    print("\n--- Starting Custom Multi-Tool Agent Conversation ---")

    # Query that should trigger the dictionary_lookup tool
    await call_agent_async("What is the definition of banana?",
                                       runner=custom_multi_tool_runner,
                                       user_id=CUSTOM_MULTI_TOOL_USER_ID,
                                       session_id=CUSTOM_MULTI_TOOL_SESSION_ID)

    # Query that should trigger the text_manipulation tool (reverse)
    await call_agent_async("Reverse the text 'hello world'",
                                       runner=custom_multi_tool_runner,
                                       user_id=CUSTOM_MULTI_TOOL_USER_ID,
                                       session_id=CUSTOM_MULTI_TOOL_SESSION_ID)

    # Query that should trigger the dictionary_lookup tool (word not found)
    await call_agent_async("What does 'zeitgeist' mean?",
                                       runner=custom_multi_tool_runner,
                                       user_id=CUSTOM_MULTI_TOOL_USER_ID,
                                       session_id=CUSTOM_MULTI_TOOL_SESSION_ID)

    # Query that should trigger the text_manipulation tool (uppercase)
    await call_agent_async("Make the following text uppercase: 'this is a test'",
                                       runner=custom_multi_tool_runner,
                                       user_id=CUSTOM_MULTI_TOOL_USER_ID,
                                       session_id=CUSTOM_MULTI_TOOL_SESSION_ID)

    # Query that might not trigger a tool (general question)
    await call_agent_async("Tell me a fun fact.",
                                       runner=custom_multi_tool_runner,
                                       user_id=CUSTOM_MULTI_TOOL_USER_ID,
                                       session_id=CUSTOM_MULTI_TOOL_SESSION_ID)


    print("\n--- Custom Multi-Tool Agent Conversation Finished ---")

# Await the execution of the conversation function
await run_custom_multi_tool_conversation()


--- Starting Custom Multi-Tool Agent Conversation ---

>>> User Query: What is the definition of banana?


--- Tool: text_manipulation called for text: 'hello world', operation: 'reverse' ---


--- Tool: dictionary_lookup called for word: banana ---
<<< Agent Response: A long, curved fruit which grows in clusters and has soft pulpy flesh and yellow skin when ripe.


>>> User Query: Reverse the text 'hello world'


--- Tool: text_manipulation called for text: 'hello world', operation: 'reverse' ---
<<< Agent Response: dlrow olleh


>>> User Query: What does 'zeitgeist' mean?


--- Tool: dictionary_lookup called for word: zeitgeist ---
<<< Agent Response: I'm sorry, I couldn't find a definition for "zeitgeist" in my current dictionary.


>>> User Query: Make the following text uppercase: 'this is a test'


--- Tool: text_manipulation called for text: 'this is a test', operation: 'uppercase' ---
<<< Agent Response: THIS IS A TEST


>>> User Query: Tell me a fun fact.
<<< Agent Response: I do not have access to external knowledge bases, including fun fact databases.  Therefore, I cannot provide you with a fun fact.


--- Custom Multi-Tool Agent Conversation Finished ---


# Structured Output in ADK

In [ ]:
# @title Define a Pydantic model
from pydantic import BaseModel

# Define the structured output schema using Pydantic
class ProductReview(BaseModel):
    """Represents a product review with a title and body."""
    title: str
    body: str

print("ProductReview Pydantic model defined.")

ProductReview Pydantic model defined.


In [ ]:
#  @title Define agent with output schema
structured_output_agent = Agent(
    name="product_review_agent",
    model="gemini-1.5-flash-latest",
    instruction=(
        "You are an assistant that summarizes user feedback into a structured product review. "
        "When the user provides feedback about a product, extract the key points and format them "
        "into a review with a concise title and a detailed body that conforms to the ProductReview schema."
    ),
    output_schema=ProductReview,
    output_key="product_review"
)

print(f"Agent '{structured_output_agent.name}' created using model '{structured_output_agent.model}' with ProductReview output schema and output_key.")

Agent 'product_review_agent' created using model 'gemini-1.5-flash-latest' with ProductReview output schema and output_key.


In [ ]:
# @title Set up session service and runner
session_service = InMemorySessionService()

STRUCTURED_OUTPUT_APP_NAME = "structured_output_agent_app"
STRUCTURED_OUTPUT_USER_ID = "structured_output_user_1"
STRUCTURED_OUTPUT_SESSION_ID = "structured_output_session_001"

structured_output_session = await session_service.create_session(
    app_name=STRUCTURED_OUTPUT_APP_NAME,
    user_id=STRUCTURED_OUTPUT_USER_ID,
    session_id=STRUCTURED_OUTPUT_SESSION_ID
)
print(f"Session created for structured output agent: App='{STRUCTURED_OUTPUT_APP_NAME}', User='{STRUCTURED_OUTPUT_USER_ID}', Session='{STRUCTURED_OUTPUT_SESSION_ID}'")

structured_output_runner = Runner(
    agent=structured_output_agent,
    app_name=STRUCTURED_OUTPUT_APP_NAME,
    session_service=session_service
)
print(f"Runner created for structured output agent '{structured_output_runner.agent.name}'.")

Session created for structured output agent: App='structured_output_agent_app', User='structured_output_user_1', Session='structured_output_session_001'
Runner created for structured output agent 'product_review_agent'.


In [ ]:
# @title Inference

from google.genai import types

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # Uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      if event.is_final_response():
          if event.content and event.content.parts:
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate:
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break

  print(f"<<< Agent Response: {final_response_text}")


async def run_structured_output_conversation():
    print("\n--- Starting Structured Output Agent Conversation ---")

    await call_agent_async("I just used the new ergonomic mouse and it is incredibly comfortable and precise. The battery life is also amazing!",
                                       runner=structured_output_runner,
                                       user_id=STRUCTURED_OUTPUT_USER_ID,
                                       session_id=STRUCTURED_OUTPUT_SESSION_ID)


    print("\n--- Structured Output Agent Conversation Finished ---")

# Await the execution of the conversation function
await run_structured_output_conversation()


--- Starting Structured Output Agent Conversation ---

>>> User Query: I just used the new ergonomic mouse and it is incredibly comfortable and precise. The battery life is also amazing!
<<< Agent Response: {"title": "Ergonomic Mouse Review", "body": "This ergonomic mouse is incredibly comfortable and precise.  The battery life is amazing!"}

--- Structured Output Agent Conversation Finished ---


In [ ]:
# @title Process the structured output

# Access the latest session state
latest_session = await session_service.get_session(
    app_name=STRUCTURED_OUTPUT_APP_NAME,
    user_id=STRUCTURED_OUTPUT_USER_ID,
    session_id=STRUCTURED_OUTPUT_SESSION_ID
)

print("\n--- Latest Session State ---")

print(latest_session)

structured_output = None
OUTPUT_KEY = "product_review" # The output_key defined in the agent

structured_output = latest_session.state[OUTPUT_KEY]

print("\n--- Structured Output ---")
print(structured_output)

if structured_output:
    print("\nAttempting final parse with Pydantic:")
    try:
        product_review_pydantic = ProductReview(**structured_output)
        print("Successfully parsed with Pydantic:")
        print(product_review_pydantic)
    except Exception as e:
        print(f"Failed to parse with Pydantic: {e}")
        print("Found JSON but it did not match the ProductReview schema.")
else:
    print("\nNo structured output (JSON matching schema) found using the output_key or in fallback checks.")


--- Latest Session State ---
id='structured_output_session_001' app_name='structured_output_agent_app' user_id='structured_output_user_1' state={'product_review': {'title': 'Ergonomic Mouse Review', 'body': 'This ergonomic mouse is incredibly comfortable and precise.  The battery life is amazing!'}} events=[Event(content=Content(
  parts=[
    Part(
      text='I just used the new ergonomic mouse and it is incredibly comfortable and precise. The battery life is also amazing!'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, invocation_id='e-8876be43-79f7-4c26-9523-e4127ee56288', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}), long_running_tool_ids=None, branch=None, id='f3dca5f1-f786-492a-bff1-054b90ba4f73', timestamp=1753269075.859617),

# LiteLLM TEST

In [ ]:
import os
from google.adk import Agent
import random

# Get the OPEN AI API key from environment variables
openai_api_key = os.environ.get("OPENAI_API_KEY")

# Set up the model using LiteLLM with OpenRouter
model = LiteLlm(
    model="gpt-3.5-turbo",
    api_key=openai_api_key
)

def get_ai_engineer_joke() -> dict:
    """
    Returns a random AI engineer joke from a curated list.
    Use this when the user asks for an AI-related joke.
    """
    jokes = [
        "Why did the neural network break up with the decision tree? It couldn’t handle its lack of depth.",
        "Why do AI engineers make terrible secret agents? Because they always leave a trace!",
        "How does an AI tell someone they’re wrong? With a high confidence score and low accuracy.",
        "Why was the LLM always tired? It had too many parameters to sleep on.",
        "How do you comfort an overfitting model? You tell it, 'It's not your data, it's your complexity.'",
        "Why did the AI fail its driving test? It couldn’t handle unstructured environments.",
        "Why do transformers love attention? Because it’s all they need!",
        "Why don’t AI models gossip? Because their training data is confidential.",
        "Why did the engineer talk to the model during inference? Because they misunderstood 'zero-shot'.",
        "Why was the dataset feeling down? It needed some normalization."
    ]
    selected_joke = random.choice(jokes)
    return {
        "joke": selected_joke
    }

ai_engineer_joke_agent = Agent(
    name="ai_engineer_joke_agent",
    model=model,
    description="An agent that tells AI engineer jokes",
    instruction="You are a witty assistant that tells jokes only about AI engineers, models, and data. Use only the tool get_ai_engineer_joke to tell a joke.",
    tools=[FunctionTool(get_ai_engineer_joke)]
)

In [ ]:
# @title Setup Session Service and Runner for Joke Agent

# Reuse the InMemorySessionService instance or create a new one if needed
# session_service = InMemorySessionService() # Uncomment if starting fresh

# Define constants for identifying the interaction context for the joke agent
JOKE_APP_NAME = "joke_agent_app"
JOKE_USER_ID = "joke_user_1"
JOKE_SESSION_ID = "joke_session_001" # Using a fixed ID for simplicity

# Create a session for the joke agent
joke_session = await session_service.create_session(
    app_name=JOKE_APP_NAME,
    user_id=JOKE_USER_ID,
    session_id=JOKE_SESSION_ID
)
print(f"Session created for joke agent: App='{JOKE_APP_NAME}', User='{JOKE_USER_ID}', Session='{JOKE_SESSION_ID}'")

# Create a runner for the joke agent
joke_runner = Runner(
    agent=ai_engineer_joke_agent, # The joke agent we want to run
    app_name=JOKE_APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for joke agent '{joke_runner.agent.name}'.")

Session created for joke agent: App='joke_agent_app', User='joke_user_1', Session='joke_session_001'
Runner created for joke agent 'ai_engineer_joke_agent'.
